# Приложение: прогнозирование характеристик композитных материалов

**Задача 3 ВКР** — приложение, выдающее прогнозные значения, полученные в результате
обучения моделей (основной ноутбук `VKR_Composites.ipynb`).

Пользователь задаёт **10 параметров** материала, приложение возвращает **3 прогноза**:

| Выход | Модель |
|---|---|
| Модуль упругости при растяжении, ГПа | Random Forest (`models/ml_modul_uprugosti.joblib`) |
| Прочность при растяжении, МПа | Random Forest (`models/ml_prochnost.joblib`) |
| Соотношение матрица–наполнитель | Нейросеть Keras (`models/nn_mf.keras`) |

Под капотом по плотности композита определяется тип армирующего волокна и вычисляются
физические признаки (правило смесей) — модуль `composite_physics.py`.

> **Запуск:** выполните все ячейки; интерактивная форма появится в последней ячейке
> (требуется окружение с включёнными ipywidgets — Jupyter Notebook/Lab).
> Альтернатива — веб-приложение Streamlit: `streamlit run app.py`.

In [2]:
# Загрузка обученных моделей и масштабаторов
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

tf.get_logger().setLevel('ERROR')

with open('models/artifacts.json', encoding='utf-8') as f:
    ARTIFACTS = json.load(f)

FEATURES = ARTIFACTS['features']          # 17 признаков для моделей задачи 1
NN_FEATURES = ARTIFACTS['nn_features']    # 12 признаков для нейросети
TARGETS = ARTIFACTS['targets']

model_E = joblib.load('models/ml_modul_uprugosti.joblib')
model_sigma = joblib.load('models/ml_prochnost.joblib')
scaler_ml = joblib.load('models/scaler_features.joblib')
scaler_X_nn = joblib.load('models/scaler_nn_X.joblib')
scaler_y_nn = joblib.load('models/scaler_nn_y.joblib')
model_nn = tf.keras.models.load_model('models/nn_mf.keras')

print('Модели загружены:', ARTIFACTS['best_model'], '+ Keras', tf.__version__)

Модели загружены: Случайный лес + Keras 2.21.0


In [3]:
# Физическая модель: тип волокна и инженерные признаки из плотности композита
from composite_physics import (FIBERS_DF, infer_fiber_type, engineered_features,
                               BASE_INPUTS, TARGETS)

# Диапазоны входных параметров — по исходным данным
_df = pd.read_excel('X_bp.xlsx', index_col=0).join(
    pd.read_excel('X_nup.xlsx', index_col=0), how='inner')
INPUT_RANGES = {c: (_df[c].min(), _df[c].max(), float(_df[c].median()))
                for c in BASE_INPUTS}
INPUT_RANGES['Угол нашивки, град'] = (0.0, 90.0, 0.0)
del _df
print('Диапазоны входных параметров заданы.')

Диапазоны входных параметров заданы.


In [4]:
def predict_all(user_input: dict) -> dict:
    """Прогноз трёх характеристик композита по 10 входным параметрам."""
    eng = engineered_features(user_input['Плотность, кг/м3'])
    row = {**user_input,
           **{k: eng[k] for k in ['fiber_type_id', 'V_f', 'E_f', 'sigma_f', 'rho_f', 'E_1', 'E_2']}}
    x_ml = pd.DataFrame(scaler_ml.transform(pd.DataFrame([row])[FEATURES]), columns=FEATURES)
    e_pred = float(model_E.predict(x_ml)[0])
    s_pred = float(model_sigma.predict(x_ml)[0])

    row_nn = {**user_input, TARGETS[0]: e_pred, TARGETS[1]: s_pred}
    x_nn = pd.DataFrame(scaler_X_nn.transform(pd.DataFrame([row_nn])[NN_FEATURES]),
                        columns=NN_FEATURES)
    mf_pred = float(scaler_y_nn.inverse_transform(
        model_nn.predict(x_nn, verbose=0)).ravel()[0])

    return {'Модуль упругости при растяжении, ГПа': round(e_pred, 2),
            'Прочность при растяжении, МПа': round(s_pred, 1),
            'Соотношение матрица-наполнитель': round(mf_pred, 3),
            'Тип волокна (по плотности)': eng['fiber_name'],
            'Объёмная доля волокна V_f': round(eng['V_f'], 3)}

## Демонстрация работы приложения

Прогнозы для трёх контрольных составов из датасета (значения входных параметров взяты
из исходных файлов X_bp + X_nup):

In [5]:
df_bp_demo = pd.read_excel('X_bp.xlsx', index_col=0)
df_nup_demo = pd.read_excel('X_nup.xlsx', index_col=0)
demo_rows = []
for i in (0, 500, 1000):
    row = {c: df_bp_demo.iloc[i][c] for c in BASE_INPUTS if c in df_bp_demo.columns}
    row.update({c: df_nup_demo.iloc[i][c] for c in BASE_INPUTS if c in df_nup_demo.columns})
    demo_rows.append(row)

pd.DataFrame([predict_all(d) for d in demo_rows])

I0000 00:00:1793119371.655545   73029 service.cc:153] XLA service 0x77814c02e3d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793119371.655578   73029 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2060 SUPER, Compute Capability 7.5 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.3.0)
I0000 00:00:1793119371.662835   73029 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1793119371.694952   73029 cuda_dnn.cc:461] Loaded cuDNN version 90300
I0000 00:00:1793119372.141568   73029 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


,"Модуль упругости при растяжении, ГПа","Прочность при растяжении, МПа",Соотношение матрица-наполнитель,Тип волокна (по плотности),Объёмная доля волокна V_f
0,41.62,1002.3,2.853,AR-glass,0.561
1,37.71,904.3,2.242,AR-glass,0.500
2,7.05,996.5,2.759,AR-glass,0.554


## Интерактивная форма

Задайте параметры материала и нажмите **«Рассчитать прогноз»**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

SHORT = {
    'Плотность, кг/м3': 'Плотность, кг/м³',
    'модуль упругости, ГПа': 'Модуль упругости волокна, ГПа',
    'Количество отвердителя, м.%': 'Количество отвердителя, м.%',
    'Содержание эпоксидных групп,%_2': 'Содержание эпоксидных групп, %',
    'Температура вспышки, С_2': 'Температура вспышки, °C',
    'Поверхностная плотность, г/м2': 'Поверхностная плотность, г/м²',
    'Потребление смолы, г/м2': 'Потребление смолы, г/м²',
    'Угол нашивки, град': 'Угол нашивки, град',
    'Шаг нашивки': 'Шаг нашивки, мм',
    'Плотность нашивки': 'Плотность нашивки',
}

input_widgets = {}
for col in BASE_INPUTS:
    lo, hi, mid = INPUT_RANGES[col]
    if col == 'Угол нашивки, град':
        input_widgets[col] = widgets.SelectionSlider(
            options=[0, 90], value=int(mid), description=SHORT[col],
            continuous_update=False, layout=widgets.Layout(width='420px'))
    elif float(hi - lo) <= 40:
        step = 0.1
        input_widgets[col] = widgets.FloatSlider(
            min=round(lo, 2), max=round(hi, 2), step=step, value=round(mid, 2),
            description=SHORT[col], continuous_update=False,
            layout=widgets.Layout(width='420px'), readout_format='.2f')
    else:
        input_widgets[col] = widgets.FloatSlider(
            min=round(lo, 1), max=round(hi, 1), step=round((hi - lo) / 200, 2), value=round(mid, 2),
            description=SHORT[col], continuous_update=False,
            layout=widgets.Layout(width='420px'), readout_format='.2f')

button = widgets.Button(description='Рассчитать прогноз', button_style='primary',
                        layout=widgets.Layout(width='300px', height='40px'))
out = widgets.Output(layout={'border': '1px solid #cccccc', 'padding': '8px'})


def on_calculate(_):
    with out:
        clear_output()
        user_input = {col: float(w.value) for col, w in input_widgets.items()}
        result = predict_all(user_input)
        print('═' * 62)
        print('ПРОГНОЗ ХАРАКТЕРИСТИК КОМПОЗИТА')
        print('═' * 62)
        print(f"  Модуль упругости при растяжении : {result['Модуль упругости при растяжении, ГПа']:>10.2f} ГПа")
        print(f"  Прочность при растяжении        : {result['Прочность при растяжении, МПа']:>10.1f} МПа")
        print(f"  Соотношение матрица-наполнитель : {result['Соотношение матрица-наполнитель']:>10.3f}")
        print('-' * 62)
        print(f"  Армирующее волокно (по плотности): {result['Тип волокна (по плотности)']}")
        print(f"  Объёмная доля волокна V_f        : {result['Объёмная доля волокна V_f']:.3f}")


button.on_click(on_calculate)

left_col = widgets.VBox(list(input_widgets.values())[:5])
right_col = widgets.VBox(list(input_widgets.values())[5:])
form = widgets.VBox([widgets.HBox([left_col, right_col]), button, out])
display(form)

### Примечания

1. Прогноз корректен в диапазонах параметров, представленных в обучающей выборке
   (диапазоны слайдеров соответствуют минимумам/максимумам данных X_bp + X_nup).
2. Соотношение «матрица–наполнитель» в приложении рассчитывается нейросетью по полному
   конвейеру: 10 параметров пользователя → ML-прогнозы модуля и прочности → нейросеть.
3. Веб-версия приложения с теми же моделями: `streamlit run app.py`.